In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [2]:
data = pd.read_excel('./data/que2/daily_data_new/data_1.xls', usecols=range(8))
for i in range(2, 12):
    data_read = pd.read_excel('./data/que2/daily_data_new/data_' + str(i) + '.xls', usecols=[0,1,2,3,4,5,6,7])
    data = pd.concat([data,data_read],ignore_index=True)
    del data_read
data.columns = ['stk', 'date', 'close', 'fshare', 'tshare', 'monret', 'monrf', 'pe']
data.dropna(inplace = True)
data

,stk,date,close,fshare,tshare,monret,monrf,pe
0,2,2001-01-19,14.93,509216805.0,398711877.0,0.0672,0.001650,31.27
1,2,2001-02-28,13.77,509216805.0,398711877.0,-0.0777,0.001650,28.84
2,2,2001-03-30,15.13,509216805.0,398711877.0,0.0988,0.001650,31.69
3,2,2001-04-30,14.58,509216805.0,398711877.0,-0.0364,0.001650,27.95
4,2,2001-05-31,14.20,509216805.0,398711877.0,-0.0261,0.001650,27.22
...,...,...,...,...,...,...,...,...
539492,873305,2022-03-16,25.00,0.0,0.0,0.0000,0.001971,13.48
539501,873305,2022-12-30,10.12,44819000.0,44819000.0,-0.5952,0.001931,12.32
539535,873339,2022-12-30,5.01,220203800.0,220203800.0,-0.1494,0.001931,80.18
539560,873527,2022-11-30,9.05,60042700.0,60042700.0,-0.0300,0.001676,4.98


In [3]:
data['date'] = pd.to_datetime(data['date'])
data['yearmonth'] = data['date'].dt.strftime('%Y%m').astype(int)
data['stksize'] = data['close']*data['fshare']
data['stkep'] = 1/data['pe']
data['monexcret'] = data['monret'] - data['monrf']
# 只针对stksize和stkep两列
data.dropna(inplace = True, subset=['stksize', 'stkep'])
data

,stk,date,close,fshare,tshare,monret,monrf,pe,yearmonth,stksize,stkep,monexcret
0,2,2001-01-19,14.93,509216805.0,398711877.0,0.0672,0.001650,31.27,200101,7.602607e+09,0.031980,0.065550
1,2,2001-02-28,13.77,509216805.0,398711877.0,-0.0777,0.001650,28.84,200102,7.011915e+09,0.034674,-0.079350
2,2,2001-03-30,15.13,509216805.0,398711877.0,0.0988,0.001650,31.69,200103,7.704450e+09,0.031556,0.097150
3,2,2001-04-30,14.58,509216805.0,398711877.0,-0.0364,0.001650,27.95,200104,7.424381e+09,0.035778,-0.038050
4,2,2001-05-31,14.20,509216805.0,398711877.0,-0.0261,0.001650,27.22,200105,7.230879e+09,0.036738,-0.027750
...,...,...,...,...,...,...,...,...,...,...,...,...
539492,873305,2022-03-16,25.00,0.0,0.0,0.0000,0.001971,13.48,202203,0.000000e+00,0.074184,-0.001971
539501,873305,2022-12-30,10.12,44819000.0,44819000.0,-0.5952,0.001931,12.32,202212,4.535683e+08,0.081169,-0.597131
539535,873339,2022-12-30,5.01,220203800.0,220203800.0,-0.1494,0.001931,80.18,202212,1.103221e+09,0.012472,-0.151331
539560,873527,2022-11-30,9.05,60042700.0,60042700.0,-0.0300,0.001676,4.98,202211,5.433864e+08,0.200803,-0.031676


In [4]:
# 所有年月信息
uym = np.unique(data['yearmonth'].values)
print(len(uym))
uym

264


array([200101, 200102, 200103, 200104, 200105, 200106, 200107, 200108,
       200109, 200110, 200111, 200112, 200201, 200202, 200203, 200204,
       200205, 200206, 200207, 200208, 200209, 200210, 200211, 200212,
       200301, 200302, 200303, 200304, 200305, 200306, 200307, 200308,
       200309, 200310, 200311, 200312, 200401, 200402, 200403, 200404,
       200405, 200406, 200407, 200408, 200409, 200410, 200411, 200412,
       200501, 200502, 200503, 200504, 200505, 200506, 200507, 200508,
       200509, 200510, 200511, 200512, 200601, 200602, 200603, 200604,
       200605, 200606, 200607, 200608, 200609, 200610, 200611, 200612,
       200701, 200702, 200703, 200704, 200705, 200706, 200707, 200708,
       200709, 200710, 200711, 200712, 200801, 200802, 200803, 200804,
       200805, 200806, 200807, 200808, 200809, 200810, 200811, 200812,
       200901, 200902, 200903, 200904, 200905, 200906, 200907, 200908,
       200909, 200910, 200911, 200912, 201001, 201002, 201003, 201004,
      

In [5]:
class sort_portfolio:
    def __init__(self, data, months, gnum):
        self.data = data
        self.months = months
        self.gnum = gnum
        
    def data_months(self):
        dm = self.data.loc[self.data['yearmonth'] == self.months[0], ['stk', 'stksize', 'stkep']]
        dm.dropna(inplace = True)
        for i in range(1, len(self.months)):
            ind = self.data['yearmonth'] == self.months[i]
            dm = pd.merge(left = dm,
                         right = self.data.loc[ind, ['stk', 'monexcret']],
                         on='stk',
                         how='left',
                         sort=True)
            # sort=True指定了合并时按照合并键进行排序。
        dm.columns = ['stk', 'size6', 'ep6', 'ret7', 'ret8', 'ret9', 'ret10', 'ret11', 
                      'ret12', 'retn1', 'retn2', 'retn3', 'retn4', 'retn5', 'retn6']
        return dm
    
    def sort_single_ind(self):
        L = np.sum(self.data['yearmonth'] == self.months[0])
        n = np.fix(L/self.gnum).astype(int) # 使用np.fix函数向下取整得到每个分组的整数股票数量
        x = np.ones(L) # 初始化加权索引数组,数组中的所有元素都初始化为1。这个数组x将用于存储每个股票的加权索引值。
        i = 0
        # 生成一个加权索引数组x，用于根据单一排序指标（例如市值）对股票进行加权排序
        while i < self.gnum:
            # 由于这是最后一个分组，所以只需要对从i*n到数组末尾的股票进行操作
            if i == self.gnum-1:
                x[i*n:] = x[i*n:]*i
            # 如果当前分组不是最后一个分组，这行代码会将该分组内所有股票的加权索引值设置为i的倍数
            else:
                x[i*n:(i+1)*n] = x[i*n:(i+1)*n]*i
            i = i+1
        ssi = x.astype(int) # 将加权索引数组x中的浮点数值转换为整数类型
        return ssi
    
    def sort_double_ind(self):
        #生成一个基于双重排序指标的加权索引数组
        L = np.sum(self.data['yearmonth'] == self.months[0])
        # 计算每个大分组（l）和小分组（n）应该有多少股票
        l = np.fix(L/self.gnum).astype(int)
        n = np.fix(L/(self.gnum**2)).astype(int)
        x = np.ones(L)
        i = 0
        while i < self.gnum:
            j = 0
            while j < self.gnum:
                if j == self.gnum-1:
                    if i == self.gnum-1:
                        x[(i*l+j*n):] = x[(i*l+j*n):]*j
                    else:
                        x[(i*l+j*n):((i+1)*l)] = x[(i*l+j*n):((i+1)*l)]*j
                else:
                    x[(i*l+j*n):(i*l+(j+1)*n)] = x[(i*l+j*n):(i*l+(j+1)*n)]*j
                j=j+1
            i=i+1
        sdi = x.astype(int)
        return sdi
    
    def sequence_sort(self):
        # 将股票数据按照单一指标和双重指标进行排序
        dm = self.data_months()
        ssi = self.sort_single_ind()
        sdi = self.sort_double_ind()
        dm.sort_values(by=['size6'], ascending=True, inplace=True) # 按市值升序排序
        dm['sinsort'] = ssi # 添加单一指标排序列
        dm.sort_values(by=['sinsort', 'ep6'], ascending=[True, True], inplace=True) # 根据单一指标和市盈率倒数排序
        dm['dousort'] = sdi #添加双重指标排序列
        return dm
    
    def sequence_sort_mreturn(self):
        sp = self.sequence_sort()
        # ret7（代表某个月份的月超额收益率）
        spmreturn = sp.loc[:, ['ret7', 'sinsort', 'dousort']].dropna().groupby(
                    by=['sinsort', 'dousort'])['ret7'].mean()
        # 计算结果是一个Series spmreturn，它的索引是sinsort和dousort的唯一组合
        lret = ['ret8', 'ret9', 'ret10', 'ret11', 'ret12', 'retn1', 'retn2', 'retn3', 'retn4', 'retn5', 'retn6']
        for i in lret:
            a = sp.loc[:, [i, 'sinsort', 'dousort']].dropna().groupby(
                by=['sinsort', 'dousort'])[i].mean()
            spmreturn = pd.concat([spmreturn, a], axis=1)
        spmreturn['mret'] = spmreturn.apply(lambda x: x.mean(), axis=1)
        # 新列mret，它代表每个排序组合在所有月份的平均收益率
        return spmreturn

In [6]:
print(uym[5:5+13])
sp = sort_portfolio(data, uym[5:5+13], 5)
sp.data_months()  
# 获取到每支股票在6月的市值，以及其后12个月的月超额收益率

[200106 200107 200108 200109 200110 200111 200112 200201 200202 200203
 200204 200205 200206]


,stk,size6,ep6,ret7,ret8,ret9,ret10,ret11,ret12,retn1,retn2,retn3,retn4,retn5,retn6
0,2,7.541501e+09,0.035224,-0.06785,0.06705,0.00995,-0.09375,0.00505,-0.01275,-0.07435,0.019414,0.001775,-0.021925,-0.091625,0.167575
1,4,2.437003e+09,0.010515,-0.15745,0.02895,-0.11845,-0.12635,0.12335,0.00155,-0.05385,0.065414,-0.019425,-0.000525,0.039175,-0.149725
2,6,3.535067e+09,0.036751,-0.05615,0.00665,-0.04825,-0.09215,0.01065,-0.03125,-0.10125,0.044214,0.029075,0.030875,-0.114225,0.295275
3,8,1.735270e+09,-0.001717,-0.26355,0.03345,-0.13495,-0.12085,0.06235,-0.18355,-0.16635,0.116514,0.439075,NaN,NaN,NaN
4,9,7.085606e+09,0.004614,-0.18025,-0.11205,-0.04975,0.00025,0.01585,-0.06845,-0.14275,0.036514,0.326575,0.019275,-0.121525,0.467775
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
827,600893,2.266890e+09,-0.004342,-0.15195,-0.03215,0.01345,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
828,600894,6.175620e+09,0.015267,-0.05165,-0.02885,-0.16225,-0.05525,0.09025,-0.07455,-0.15595,-0.010486,0.224175,0.206275,-0.085525,0.146375
829,600895,8.668655e+09,0.017727,-0.12785,-0.04485,-0.05645,-0.13475,0.09365,0.00415,0.00695,0.006214,0.277075,-0.027325,-0.083325,0.320875
830,600897,4.093200e+09,0.012569,0.04585,-0.03375,-0.17535,0.02035,0.07075,-0.11225,-0.16565,0.032214,0.077975,0.053975,-0.014525,0.143175


In [7]:
sp.sequence_sort()

,stk,size6,ep6,ret7,ret8,ret9,ret10,ret11,ret12,retn1,retn2,retn3,retn4,retn5,retn6,sinsort,dousort
119,566,2.213698e+09,-0.050994,-0.08665,-0.00365,-0.12375,-0.07695,0.03045,-0.31345,-0.13355,0.132414,0.042675,0.240175,-0.044925,-0.001425,0,0
669,600684,2.014414e+09,-0.050000,-0.04715,0.03925,-0.12315,-0.05165,0.02295,-0.10985,0.01555,0.009214,0.046275,0.045175,-0.123125,0.126075,0,0
116,560,1.635648e+09,-0.046707,-0.14465,0.05685,-0.18645,0.01275,0.11885,-0.19035,-0.19565,0.149414,0.041475,NaN,NaN,NaN,0,0
406,600082,2.440305e+09,-0.045600,-0.14755,0.04195,-0.04955,-0.10665,0.06825,-0.15645,-0.20435,0.028514,NaN,NaN,NaN,NaN,0,0
207,723,1.989289e+09,-0.034674,-0.15395,0.00005,-0.23145,0.06595,0.00735,-0.22175,-0.14595,0.055114,0.043775,0.172875,-0.126825,0.141975,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
232,778,6.369770e+09,0.046707,-0.09515,0.08565,-0.12225,0.06635,0.11705,-0.00095,-0.03255,0.012914,0.133575,0.024875,-0.114925,0.147975,4,4
196,709,1.165647e+10,0.050736,-0.09925,-0.07305,-0.02635,0.00575,-0.00605,-0.02985,-0.03065,0.022014,0.093775,0.006975,-0.104325,0.158275,4,4
433,600126,7.369754e+09,0.051387,-0.13825,-0.03305,-0.05615,-0.05365,0.01475,-0.04415,-0.06165,0.022714,0.055975,0.022175,-0.128125,0.142375,4,4
492,600207,9.878000e+09,0.052383,-0.25555,0.00015,-0.01055,0.14565,-0.01525,0.10355,-0.21795,-0.086286,-0.021525,0.001975,-0.234525,0.148775,4,4


In [9]:
sp.sequence_sort_mreturn()

ret7      ret8      ret9     ret10     ret11     ret12  \
sinsort dousort                                                               
0       0       -0.144886  0.005371 -0.100992 -0.080250  0.075450 -0.111672   
        1       -0.139029 -0.008274 -0.068495 -0.082180  0.074789 -0.081683   
        2       -0.142798 -0.021826 -0.060571 -0.077908  0.059253 -0.062253   
        3       -0.144286 -0.013259 -0.062038 -0.076598  0.060850 -0.048986   
        4       -0.136985 -0.028697 -0.038694 -0.061147  0.043171 -0.050794   
1       0       -0.132880 -0.020468 -0.100880 -0.040780  0.054568 -0.137526   
        1       -0.155089 -0.033841 -0.055589 -0.054071  0.044817 -0.056777   
        2       -0.137447 -0.037141 -0.056644 -0.082086  0.065038 -0.062044   
        3       -0.118962 -0.034741 -0.055553 -0.068756  0.063235 -0.041508   
        4       -0.141056 -0.028144 -0.072985 -0.071644  0.062306 -0.060741   
2       0       -0.125408 -0.037189 -0.061183 -0.035023  0.036314 -0.083341   
        1       -0.156653 -0.047905 -0.074032 -0.058298  0.051617 -0.086541   
        2       -0.137144 -0.027859 -0.050483 -0.075277  0.037574 -0.065868   
        3       -0.139077 -0.050356 -0.051520 -0.051268  0.050850 -0.071159   
        4       -0.121059 -0.048438 -0.049162 -0.046888  0.043332 -0.038574   
3       0       -0.134035 -0.050438 -0.061432 -0.044941  0.053944 -0.078392   
        1       -0.139017 -0.033547 -0.039226 -0.091002  0.031311 -0.073126   
        2       -0.145068 -0.021768 -0.048350 -0.062826  0.064195 -0.083492   
        3       -0.141768 -0.050656 -0.054274 -0.054686  0.054971 -0.068232   
        4       -0.122862 -0.033744 -0.047550 -0.054000  0.050697 -0.036379   
4       0       -0.129047 -0.049662 -0.073395 -0.051144  0.034862 -0.058829   
        1       -0.120347 -0.056795 -0.035265 -0.027759  0.039653 -0.081844   
        2       -0.152202 -0.025268 -0.024386 -0.036486  0.036759 -0.061783   
        3       -0.147102 -0.033171 -0.036026 -0.037720  0.043741 -0.050250   
        4       -0.128786 -0.014019 -0.031031 -0.029436  0.020344 -0.009389   

                    retn1     retn2     retn3     retn4     retn5     retn6  \
sinsort dousort                                                               
0       0       -0.170197  0.057805  0.099705  0.085371 -0.093912  0.095740   
        1       -0.148032  0.037996  0.108360  0.079584 -0.076100  0.073381   
        2       -0.133859  0.026723  0.071560  0.055736 -0.080016  0.088842   
        3       -0.136956  0.020156  0.096039  0.066911 -0.089077  0.110436   
        4       -0.105300  0.021770  0.071719  0.061463 -0.093122  0.092551   
1       0       -0.149377  0.052481  0.105856  0.073660 -0.062305  0.122279   
        1       -0.150338  0.033023  0.088760  0.073696 -0.082884  0.104884   
        2       -0.120353  0.026977  0.060269  0.073637 -0.086434  0.100425   
        3       -0.116129  0.023426  0.100975  0.025875 -0.091919  0.091960   
        4       -0.108891  0.038493  0.084596  0.068975 -0.092054  0.113540   
2       0       -0.148523  0.029784  0.079869  0.076262 -0.077118  0.090825   
        1       -0.134577  0.032332  0.087030  0.050906 -0.084972  0.107800   
        2       -0.114123  0.019162  0.068911  0.036569 -0.083795  0.105696   
        3       -0.107998  0.039029  0.073360  0.045333 -0.085037  0.084945   
        4       -0.093432  0.035732  0.045313  0.026475 -0.109434  0.142245   
3       0       -0.121844  0.030987  0.089654  0.048188 -0.084788  0.107906   
        1       -0.093589  0.022181  0.065351  0.052099 -0.106543  0.108896   
        2       -0.136126  0.039129  0.085323  0.032357 -0.102343  0.131384   
        3       -0.115265  0.037056  0.067733  0.037360 -0.088501  0.159805   
        4       -0.081338  0.025896  0.060913  0.026740 -0.091813  0.145460   
4       0       -0.105669  0.028230  0.066687  0.046094 -0.069672  0.146732   
        1       -0.093214  0.027078  0.072230  0.035405 -0.080883  0.1

In [10]:
meanret = []
lcname = []
for i in range(5, 234, 12):
    if len(uym[i:i+13]) == 13:
        lcname.append(str(uym[i]))
        sp = sort_portfolio(data, uym[i:i+13], 5)
        spmreturn = sp.sequence_sort_mreturn()
        if len(meanret) == 0:
            meanret = spmreturn['mret']
        else:
            meanret = pd.concat([meanret, spmreturn['mret']], axis=1)
meanret.columns = lcname
meanret

200106    200206    200306    200406    200506    200606  \
sinsort dousort                                                               
0       0       -0.023539 -0.025819 -0.019413 -0.037364  0.035058  0.092330   
        1       -0.019140 -0.030610 -0.022117 -0.032412  0.044596  0.077170   
        2       -0.023093 -0.023900 -0.020846 -0.041251  0.034674  0.073494   
        3       -0.018067 -0.025037 -0.016940 -0.033224  0.043057  0.077337   
        4       -0.018672 -0.020788 -0.012383 -0.023497  0.040125  0.074142   
1       0       -0.019614 -0.029265 -0.023758 -0.035305  0.036389  0.084675   
        1       -0.020284 -0.026535 -0.018902 -0.032520  0.043681  0.086293   
        2       -0.021317 -0.027298 -0.020556 -0.034588  0.029056  0.068333   
        3       -0.018508 -0.020778 -0.017135 -0.024027  0.040899  0.072311   
        4       -0.017301 -0.018961 -0.007446 -0.016380  0.044047  0.087118   
2       0       -0.021228 -0.030475 -0.015898 -0.041295  0.033354  0.072309   
        1       -0.026108 -0.026933 -0.017199 -0.028804  0.037781  0.064273   
        2       -0.023886 -0.023486 -0.017374 -0.029425  0.036628  0.060170   
        3       -0.021908 -0.024750 -0.013435 -0.020382  0.044221  0.072618   
        4       -0.017824 -0.016149 -0.006243 -0.017284  0.037874  0.079471   
3       0       -0.020433 -0.023147 -0.021925 -0.026695  0.040624  0.077211   
        1       -0.024684 -0.020548 -0.019519 -0.028420  0.046663  0.073427   
        2       -0.020632 -0.018422 -0.010951 -0.025234  0.050663  0.079142   
        3       -0.018038 -0.017321 -0.012006 -0.020245  0.048889  0.077407   
        4       -0.013165 -0.014161 -0.005629 -0.019841  0.031363  0.085222   
4       0       -0.017901 -0.015341 -0.029370 -0.040411  0.030619  0.083256   
        1       -0.017115 -0.021915 -0.010613 -0.025707  0.036717  0.075105   
        2       -0.013976 -0.014395 -0.009724 -0.015853  0.031834  0.090406   
        3       -0.015416 -0.012800 -0.000019 -0.010204  0.033333  0.084669   
        4       -0.010351 -0.004267  0.005651 -0.013628  0.023237  0.093854   

                   200706    200806    200906    201006    201106    201206  \
sinsort dousort                                                               
0       0        0.014819  0.053352  0.016042  0.037355 -0.013430 -0.003013   
        1        0.003159  0.046337  0.026985  0.027269 -0.017321  0.003272   
        2       -0.001694  0.050968  0.022133  0.029100 -0.016259  0.006490   
        3        0.015684  0.052735  0.024154  0.033891 -0.010809  0.000409   
        4        0.002242  0.046919  0.024553  0.034997 -0.017844 -0.003228   
1       0       -0.015572  0.040272  0.014073  0.020606 -0.027509 -0.004206   
        1       -0.004170  0.039187  0.013088  0.017730 -0.018640  0.006194   
        2       -0.010154  0.041839  0.019069  0.019812 -0.016467  0.007838   
        3       -0.015213  0.042459  0.024230  0.022734 -0.018656  0.001441   
        4        0.010993  0.045302  0.018456  0.024915 -0.017983 -0.000586   
2       0       -0.014346  0.038807  0.003668  0.018667 -0.031339 -0.008494   
        1       -0.014423  0.034682  0.008768  0.014741 -0.019216  0.005564   
        2       -0.008568  0.034896  0.020688  0.017609 -0.017090  0.007935   
        3       -0.009014  0.040445  0.023919  0.021040 -0.018909 -0.003514   
        4       -0.003458  0.045618  0.016865  0.022446 -0.024044 -0.005872   
3       0       -0.024117  0.026995  0.004949  0.022907 -0.028752 -0.001664   
        1       -0.032020  0.022114  0.011136  0.010740 -0.020477  0.006409   
        2       -0.008869  0.018484  0.006901  0.012992 -0.018417 -0.000039   
        3       -0.012485  0.034678  0.018758  0.021940 -0.021684 -0.007145   
        4       -0.011345  0.034990  0.007582  0.022716 -0.023359 -0.006829   
4       0       -0.035988  0.022405 -0.003980  0.014223 -0.027977 -0.006630   
        1       -0.014212  0.019648 -0.006821  0.015919 -0.016691 -0

In [11]:
meanret['meanreturn'] = meanret.apply(lambda x: x.mean(), axis=1)
a = meanret['meanreturn'].values.reshape((5,5))
a

array([[0.01580497, 0.01395391, 0.01417568, 0.01743677, 0.01525365],
       [0.00730241, 0.00977825, 0.00958576, 0.01002385, 0.01429484],
       [0.00363597, 0.00506724, 0.00747783, 0.00990828, 0.0102666 ],
       [0.00371416, 0.00422354, 0.00693422, 0.00838584, 0.00889257],
       [0.00167425, 0.00454118, 0.0043627 , 0.00663729, 0.00878632]])

In [12]:
print('{:>10s} {:>10s}, {:>10s}, {:>10s}, {:>10s}, {:>10s}'.format('', 'EP1', 'EP2', 'EP3', 'EP4', 'EP5'))
for i in range(5):
    print('{:>10s} {:10.5f}, {:10.5f}, {:10.5f}, {:10.5f}, {:10.5f}'.format('SIZE'+str(i+1), 
                                                                            a[i, 0], 
                                                                            a[i, 1], 
                                                                            a[i, 2], 
                                                                            a[i, 3], 
                                                                            a[i, 4]))

                  EP1,        EP2,        EP3,        EP4,        EP5
     SIZE1    0.01580,    0.01395,    0.01418,    0.01744,    0.01525
     SIZE2    0.00730,    0.00978,    0.00959,    0.01002,    0.01429
     SIZE3    0.00364,    0.00507,    0.00748,    0.00991,    0.01027
     SIZE4    0.00371,    0.00422,    0.00693,    0.00839,    0.00889
     SIZE5    0.00167,    0.00454,    0.00436,    0.00664,    0.00879
